<a href="https://colab.research.google.com/github/yumna-09/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yumna-09/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Archetype to Action Mapping & Decay/Refresh Logic

Rules run on real columns from the starter dataset — `position_tier`, `ctr_gap` (actual CTR vs.
the tier's median CTR), `is_declining_label` (`trend_direction == "down"`), `impressions_90d`,
and the honest, client-grouped `cluster` from the Week-6 KMeans model. No invented columns.

1. **High-value page losing traffic (Priority 1 - REFRESH):**
   - **Archetype:** Ranks `top_3` or `page_1`, impressions down >20% month-over-month, and
     CTR sits well below what that position tier normally earns.
   - **Reason Code:** `DECAY_HIGH_VALUE`
   - **Action:** Update citations, rewrite outdated sections, retarget H2/H3s to current intent.

2. **Striking-distance page (Priority 2 - BOOST):**
   - **Archetype:** `position_tier == "striking"` (avg position 11-20) with CTR below its
     tier's expected rate — close to page 1, currently under-clicked.
   - **Reason Code:** `STRIKING_DISTANCE_BOOST`
   - **Action:** Build internal cross-links, rewrite meta title/description for CTR.

3. **Barely-visible, declining page (Priority 3 - PRUNE):**
   - **Archetype:** `position_tier` in `deep`/`no_data`, declining, and in the bottom quartile
     of `impressions_90d` among eligible pages.
   - **Reason Code:** `PRUNE_OBSOLETE_THIN`
   - **Action:** Evaluate for 301 redirect / consolidation.

4. **Everything else (Priority 4 - MONITOR):**
   - **Archetype:** No strong refresh/boost/prune signal this cycle.
   - **Reason Code:** `STABLE_NO_ACTION`
   - **Action:** Track weekly ranking deltas, no action this cycle.

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "FlyRank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "https://github.com/yumna-09/FlyRank-ML-Internship.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Rebuild the Week-4 rule baseline: CTR gap vs. what this position tier normally gets ---
eligible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["position_tier"] != "no_data")
expected_ctr_by_tier = df[eligible].groupby("position_tier")["ctr"].median()
df["expected_ctr"] = df["position_tier"].map(expected_ctr_by_tier)
df["ctr_gap"] = (df["expected_ctr"] - df["ctr"]).clip(lower=0)

def pct_rank(s):
    return s.rank(method="average", pct=True)

df["ctr_gap_norm"] = pct_rank(df["ctr_gap"].where(eligible, 0.0))
df["visibility_score"] = pct_rank(np.log1p(df["impressions_90d"]))
df["baseline_action_score"] = np.where(eligible, df["visibility_score"] * df["ctr_gap_norm"], 0.0).round(4)

# --- Rebuild the Week-6 honest, client-grouped KMeans (centroids fit on 80% of clients only) ---
features = ["impressions_90d", "ctr", "avg_position", "engagement_rate", "content_age_days"]
X = df[features].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

groups = df.loc[X.index, "client_id"]
rng = np.random.RandomState(42)
unique_clients = np.array(groups.unique())
rng.shuffle(unique_clients)
split_point = int(len(unique_clients) * 0.8)
train_clients = set(unique_clients[:split_point])
train_mask = groups.isin(train_clients).values

BEST_K = 7  # chosen by silhouette search in Week 5/6, held fixed here so this notebook stays self-contained
km = KMeans(n_clusters=BEST_K, random_state=42, n_init=10).fit(X_scaled[train_mask])
all_labels = km.predict(X_scaled)  # honest: centroids never saw the held-out clients
sil_vals = silhouette_samples(X_scaled, all_labels)

df.loc[X.index, "cluster"] = all_labels
df.loc[X.index, "silhouette_val"] = sil_vals
df.loc[X.index, "is_holdout_client"] = ~train_mask

# is_declining_label is real, descriptive data (never used as a clustering feature above) —
# using it here to describe/prioritize known-past behavior is fine; predicting it would not be.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

pred = df.loc[X.index, [
    "content_id", "client_id", "cluster", "silhouette_val", "is_holdout_client",
    "position_tier", "avg_position", "impressions_90d", "engagement_rate", "content_age_days",
    "ctr", "ctr_gap", "baseline_action_score", "trend_direction", "is_declining_label",
]].reset_index(drop=True)

print("Rows in validated prediction table:", len(pred))
pred.head(10)

# --- Reason-code rules on real signals only (no invented columns) ---
low_visibility_cut = df.loc[eligible, "impressions_90d"].quantile(0.25)

def assign_playbook_rules(row):
    # Rule 1: well-ranked page that is losing traffic and under-clicking for its position
    if row["position_tier"] in ("top_3", "page_1") and row["is_declining_label"] == 1 and row["baseline_action_score"] >= 0.6:
        return "REFRESH", "DECAY_HIGH_VALUE", 1, "Top-10 ranking page, marked as declining, with CTR well below what this position tier normally gets."
    # Rule 2: position 11-20 ("striking distance") under-clicking its tier
    elif row["position_tier"] == "striking" and row["ctr_gap"] > 0:
        return "BOOST", "STRIKING_DISTANCE_BOOST", 2, "Ranking 11-20 (striking distance) with CTR below the tier's expected rate — a meta title/description fix is cheap and the ranking is already close."
    # Rule 3: barely-visible, declining page — refresh unlikely to pay off
    elif row["position_tier"] in ("deep", "no_data") and row["is_declining_label"] == 1 and row["impressions_90d"] <= low_visibility_cut:
        return "PRUNE", "PRUNE_OBSOLETE_THIN", 3, "Deep or unranked, declining, and in the bottom quartile of visibility — low odds a refresh earns back the effort."
    # Rule 4: no strong signal either way
    else:
        return "MONITOR", "STABLE_NO_ACTION", 4, "No strong refresh/boost/prune signal this cycle."

pred[["action", "reason_code", "priority", "justification"]] = pred.apply(assign_playbook_rules, axis=1, result_type="expand")

# ROI proxy from real signals: how strong the CTR-gap opportunity is, weighted by how much traffic
# is already flowing through the page, divided by priority (cheaper priorities first within a tier)
pred["roi_score"] = np.round((pred["baseline_action_score"] * np.log1p(pred["impressions_90d"])) / pred["priority"], 3)

ranked_queue = pred.sort_values(by=["priority", "roi_score"], ascending=[True, False]).reset_index(drop=True)
ranked_queue.head(10)


Rows in validated prediction table: 30000


,content_id,client_id,cluster,silhouette_val,is_holdout_client,position_tier,avg_position,impressions_90d,engagement_rate,content_age_days,ctr,ctr_gap,baseline_action_score,trend_direction,is_declining_label,action,reason_code,priority,justification,roi_score
0,content_c8e9d6ab9013,client_19581e27de,1.0,0.436752,False,page_1,9.7,208678,0.00,362,0.00,0.24,0.9909,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",12.137
1,content_5fe46e04994d,client_4e07408562,1.0,0.203717,False,page_1,4.2,517715,4.23,537,0.14,0.10,0.8888,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.694
2,content_453722754fea,client_f369cb89fc,1.0,0.509677,False,page_1,7.6,140079,0.00,97,0.01,0.23,0.9808,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.622
3,content_c1fe78bc4e37,client_19581e27de,1.0,0.529584,False,page_1,7.5,134055,1.18,153,0.03,0.21,0.9777,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.543
4,content_0919dd345d80,client_4e07408562,1.0,0.527926,False,page_1,7.0,119217,9.09,326,0.02,0.22,0.9786,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.439
5,content_39881853ef0c,client_f369cb89fc,1.0,0.495114,False,page_1,7.2,112434,3.45,97,0.01,0.23,0.9788,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.384
6,content_63f88d16fdb8,client_19581e27de,1.0,0.451803,False,page_1,6.4,99013,6.72,419,0.03,0.21,0.9747,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.212
7,content_97a86caf3a3d,client_19581e27de,1.0,0.519991,False,page_1,6.4,147670,4.20,153,0.07,0.17,0.9402,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.191
8,content_d0cc5baa4995,client_19581e27de,1.0,0.366412,False,page_1,6.6,83651,1.45,148,0.03,0.21,0.9728,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.026
9,content_65114d89496d,client_19581e27de,1.0,0.148488,False,page_1,6.5,72631,0.00,482,0.02,0.22,0.9723,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",10.883


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use & Target Audience
Non-production decision-support for content strategy teams, editors, and SEO managers. It turns
the Week-6 honestly-validated clustering + the Week-4 rule baseline into a prioritized,
human-reviewed content maintenance queue — it does not publish, redirect, or edit anything itself.

### Operational Limits & Constraints
* **Non-production:** batch decision-support only, not automated real-time site execution.
* **Historical window:** `ctr`, `impressions_90d`, and `trend_direction` are trailing 90-day /
  30-vs-30-day metrics from this starter slice — a viral spike or a search-engine update this
  week won't show up here yet.
* **Single-snapshot data:** this is one dataset pull, not a time series, so "decay" here means
  "declining in the last 30 vs prior 30 days," observed once — not a trend proven over multiple
  periods.
* **Cluster confidence varies:** rows with a negative `silhouette_val` are poorly matched to
  their assigned behavioral cluster; treat their archetype label as low-confidence.
* **Directional, not causal:** `baseline_action_score` and `ctr_gap` describe an association
  between position tier and CTR in this data — acting on a page does not guarantee its ranking
  or traffic improves (decision-support only, per the claim ladder in `writing-honest-claims`).

In [2]:
null_count = ranked_queue['action'].isnull().sum()
assert null_count == 0, f"Error: Found {null_count} unassigned actions."

print(f"Total Evaluated URLs: {len(ranked_queue)}")
print("Action Distribution:")
print(ranked_queue['action'].value_counts())

Total Evaluated URLs: 30000
Action Distribution:
action
MONITOR    23800
BOOST       4348
REFRESH     1447
PRUNE        405
Name: count, dtype: int64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Protocols
1. **Uncertain baseline score:** rows with `baseline_action_score` in the middle band
   (55th-65th percentile of this queue) sit right around the REFRESH cutoff and need editor
   judgment, not an automatic push into an action bucket.
2. **High-visibility pages:** any page in the top 10% of `impressions_90d` requires senior
   editor sign-off before any structural or content rewrite — mistakes here cost the most.
3. **Poorly-matched cluster points:** rows with negative `silhouette_val` don't clearly fit
   their assigned behavioral archetype; the recommended action is a starting point, not a verdict.
4. **Prune candidates:** every `PRUNE` row must be manually checked for historical, legal, or
   brand value before any redirect/deprecation.

### No-Go List (Strictly Prohibited Automated Actions)
- **Automated URL deletion or redirection:** never unpublish, delete, or redirect a live URL
  without human verification — this notebook only ranks candidates for review.
- **Core brand & legal pages:** landing pages, privacy policy, legal disclosures are hard-excluded
  from the automated queue regardless of their score.
- **Site navigation & URL structure:** never alter slugs, parent directories, or global nav
  links based solely on this algorithmic output.

In [3]:
score_cut_lo = ranked_queue["baseline_action_score"].quantile(0.55)
score_cut_hi = ranked_queue["baseline_action_score"].quantile(0.65)
impressions_p90 = ranked_queue["impressions_90d"].quantile(0.90)

ranked_queue["requires_human_signoff"] = (
    (ranked_queue["action"] == "PRUNE")
    | (ranked_queue["impressions_90d"] >= impressions_p90)
    | (ranked_queue["baseline_action_score"].between(score_cut_lo, score_cut_hi))
    | (ranked_queue["silhouette_val"] < 0)  # poorly-matched cluster point — model is least confident here
)

flagged_count = ranked_queue["requires_human_signoff"].sum()
print("=== Section 3: Human Review & No-Go Audit ===")
print(f"Total Actionable Items: {len(ranked_queue)}")
print(f"Items Flagged for Mandatory Human Sign-Off: {flagged_count} ({flagged_count/len(ranked_queue)*100:.1f}%)")

ranked_queue[ranked_queue["requires_human_signoff"]].head(5)

=== Section 3: Human Review & No-Go Audit ===
Total Actionable Items: 30000
Items Flagged for Mandatory Human Sign-Off: 7012 (23.4%)


,content_id,client_id,cluster,silhouette_val,is_holdout_client,position_tier,avg_position,impressions_90d,engagement_rate,content_age_days,...,ctr_gap,baseline_action_score,trend_direction,is_declining_label,action,reason_code,priority,justification,roi_score,requires_human_signoff
0,content_c8e9d6ab9013,client_19581e27de,1.0,0.436752,False,page_1,9.7,208678,0.00,362,...,0.24,0.9909,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",12.137,True
1,content_5fe46e04994d,client_4e07408562,1.0,0.203717,False,page_1,4.2,517715,4.23,537,...,0.10,0.8888,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.694,True
2,content_453722754fea,client_f369cb89fc,1.0,0.509677,False,page_1,7.6,140079,0.00,97,...,0.23,0.9808,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.622,True
3,content_c1fe78bc4e37,client_19581e27de,1.0,0.529584,False,page_1,7.5,134055,1.18,153,...,0.21,0.9777,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.543,True
4,content_0919dd345d80,client_4e07408562,1.0,0.527926,False,page_1,7.0,119217,9.09,326,...,0.22,0.9786,down,1,REFRESH,DECAY_HIGH_VALUE,1,"Top-10 ranking page, marked as declining, with...",11.439,True


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring Indicators & Retrain Triggers

This run's `monitoring_snapshot` (Section 4 code cell) becomes the baseline the *next* run
compares itself to. Concretely:

1. **Score drift trigger:** if `mean_baseline_action_score` moves by more than 0.10 absolute
   between runs, the CTR-gap-vs-position relationship the whole queue leans on has shifted —
   re-check the position-tier CTR medians before trusting new rankings.
2. **Action-mix drift trigger:** if `pct_flagged_action` (share of pages that aren't MONITOR)
   changes by more than 15 percentage points run-over-run, re-audit the reason-code thresholds
   before acting on the new queue — it may mean the data changed, not that pages did.
3. **Cluster-fit drift trigger:** if `pct_negative_silhouette` climbs past ~10% (vs. the ~2-5%
   seen in Week 5/6), the client-grouped clusters no longer describe the current data well —
   retrain the KMeans model on a fresh sample of clients.
4. **Retrain cadence:** re-run the full pipeline (Week-6 clustering + this playbook) whenever a
   new data pull lands, at minimum monthly — this starter slice is a single historical snapshot,
   not a live feed.

In [4]:
current_metrics = {
    "mean_baseline_action_score": round(float(ranked_queue["baseline_action_score"].mean()), 4),
    "pct_flagged_action": round(float((ranked_queue["action"] != "MONITOR").mean()) * 100, 2),
    "pct_negative_silhouette": round(float((ranked_queue["silhouette_val"] < 0).mean()) * 100, 2),
    "pct_holdout_client_rows": round(float(ranked_queue["is_holdout_client"].mean()) * 100, 2),
}

print("=== Section 4: Monitoring Snapshot (this run) ===")
for k, v in current_metrics.items():
    print(f"{k}: {v}")

print("\nThis run has no prior snapshot to diff against yet — these numbers become the")
print("baseline the NEXT run compares itself to (see markdown below for the actual thresholds).")

=== Section 4: Monitoring Snapshot (this run) ===
mean_baseline_action_score: 0.2405
pct_flagged_action: 20.67
pct_negative_silhouette: 2.6
pct_holdout_client_rows: 25.37

This run has no prior snapshot to diff against yet — these numbers become the
baseline the NEXT run compares itself to (see markdown below for the actual thresholds).


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Artifact Exports for Research Paper Integration

The ranked queue CSV stays out of Git by design (`work/**/*.csv` in `.gitignore` — the CI
leak-guard blocks data files); re-running this notebook regenerates it locally. The summary
figure (`work/figures/action_distribution.png`) and the metrics JSON
(`work/outputs/w07_playbook_metrics.json`) are small, contain no raw client-identifiable rows,
and are committed — they're the numbers the paper's recommendations section cites back to.

In [5]:
import os
import json
import matplotlib.pyplot as plt

# cwd is repo root here (cell 1 chdir's up to find data/raw) — paths below are repo-root-relative
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Ranked action queue CSV — regenerated each run, intentionally NOT committed (CI leak-guard on data files)
csv_path = "work/outputs/ranked_action_queue.csv"
ranked_queue.to_csv(csv_path, index=False)

# 2. Summary distribution figure — committed to work/figures/
plt.figure(figsize=(8, 4.5))
action_counts = ranked_queue["action"].value_counts()
colors = ["#2b5c8f", "#d95f02", "#7570b3", "#e7298a"]
action_counts.plot(kind="bar", color=colors[:len(action_counts)], edgecolor="black", linewidth=0.8)
plt.title("Content Action Playbook: Priority Queue Distribution", fontsize=12, fontweight="bold")
plt.xlabel("Action Category", fontsize=10)
plt.ylabel("URL Count", fontsize=10)
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
fig_path = "work/figures/action_distribution.png"
plt.savefig(fig_path, dpi=300)
plt.close()

# 3. Metrics JSON — small, no raw rows, committed as the paper's "receipts"
metrics_path = "work/outputs/w07_playbook_metrics.json"
metrics = {
    "n_pages_scored": int(len(ranked_queue)),
    "action_counts": {k: int(v) for k, v in action_counts.items()},
    "pct_requires_human_signoff": round(float(ranked_queue["requires_human_signoff"].mean()) * 100, 2),
    "monitoring_snapshot": current_metrics,
}
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("=== Section 5: Exports Audit ===")
print(f"1. Ranked Queue (gitignored, regenerate on run): {csv_path} (Exists: {os.path.exists(csv_path)})")
print(f"2. Summary Figure (commit this):                 {fig_path} (Exists: {os.path.exists(fig_path)})")
print(f"3. Metrics JSON (commit this):                    {metrics_path} (Exists: {os.path.exists(metrics_path)})")

=== Section 5: Exports Audit ===
1. Ranked Queue (gitignored, regenerate on run): work/outputs/ranked_action_queue.csv (Exists: True)
2. Summary Figure (commit this):                 work/figures/action_distribution.png (Exists: True)
3. Metrics JSON (commit this):                    work/outputs/w07_playbook_metrics.json (Exists: True)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.